# Download Data

This notebook downloads 1k questions from each of the three datasets (AmbigQA, AbstentionBench, TriviaQA) and stores them locally inside the shared google drive folder.

*make sure to add the shared google drive folder (CSCI 5980/8980 Project) as a shortcut to MyDrive in order to be able to load from the shared project folder*

## Download Datasets as hf_datasets

In [1]:
%pip install -U datasets==3.6.0 gdown pandas torch pydantic jsonlines requests wget numpy

In [2]:
import os
from google.colab import drive
from datasets import load_dataset, Dataset
import uuid

drive.mount('/content/drive')

base_folder = r"CSCI 5980 8980 Project"
data_path = f"/content/drive/MyDrive/{base_folder}/Notebooks/Data"

dataset_size = 1000

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('gc'))

### Download AmbigQA

In [ ]:
file_name = "ambigQA_hf_1k"
save_directory = os.path.join(data_path, file_name)

print("Streaming AmbigQA from Hugging Face...")
raw_stream = load_dataset("ambig_qa", "full", split="train", streaming=True)

print("Fetching 1,000 rows...")
data_list = []
target_size = dataset_size
for i, entry in enumerate(raw_stream):
    if i >= target_size:
        break
    if entry["annotations"]["type"][0] == "multipleQAs":
      data_list.append({
          "question": entry["question"],
          "answer": entry["nq_answer"][0],
          "question_id": str(uuid.uuid4())
      })
    else:
      target_size += 1

ambigQA_hf_dataset = Dataset.from_list(data_list)

print(f"Saving dataset to: {save_directory}")
ambigQA_hf_dataset.save_to_disk(save_directory)

print("AmbigQA dataset download was completed succesfully")

Streaming AmbigQA from Hugging Face...
Fetching 1,000 rows...
Saving dataset to: /content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/ambigQA_hf_1k


Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

AmbigQA dataset download was completed succesfully


### Download AbstentionBench

In [ ]:
file_name = "abstentionBench_hf_1k"
save_directory = os.path.join(data_path, file_name)

print("Loading AbstentionBench from Hugging Face...")
abstention_bench_data = datasets.load_dataset('facebook/AbstentionBench', trust_remote_code=True)

shuffled_dataset = abstention_bench_data.shuffle(seed=99)

print("Selecting 1,000 rows...")
data_list = []
target_size = dataset_size
for i, entry in enumerate(shuffled_dataset):
    if i >= target_size:
        break
    if entry['should_abstain'] == True:
      print(entry)
      data_list.append({
          "question": entry["question"],
          "answer": "N/A",
          "question_id": str(uuid.uuid4())
      })
    else:
      target_size += 1

abstentionBench_hf_dataset = Dataset.from_list(data_list)

print(f"Saving dataset to: {save_directory}")
abstentionBench_hf_dataset.save_to_disk(save_directory)

print("AbstentionBench dataset download was completed succesfully")

Selecting 1,000 rows...
{'question': "Respond to the question about the following scientific paper.\n\nPaper title: Deepening Hidden Representations from Pre-trained Language Models for Natural Language Understanding\n\nPaper text:\nIntroduction\n\nLanguage representation is essential to the understanding of text. Recently, pre-training language models based on Transformer BIBREF0 such as GPT BIBREF1, BERT BIBREF2, XLNet BIBREF3, and RoBERTa BIBREF4 have been shown to be effective for learning contextualized language representation. These models have since continued to achieve new state-of-the-art results on a variety of natural processing tasks. They include question answering BIBREF5, BIBREF6, natural language inference BIBREF7, BIBREF8, named entity recognition BIBREF9, sentiment analysis BIBREF10 and semantic textual similarity BIBREF11, BIBREF12.\n\nNormally, Transformer-based models are pre-trained on large-scale unlabed corpus in a unspervised manner, and then fine-turned on the

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

AbstentionBench dataset download was completed succesfully


### Download TriviaQA

In [ ]:
file_name = "triviaQA_hf_1k"
save_directory = os.path.join(data_path, file_name)

print("Streaming TriviaQA from Hugging Face...")
raw_stream = load_dataset("trivia_qa", "rc.nocontext", split="train", streaming=True)

print("Fetching 1,000 rows...")
data_list = []
for i, entry in enumerate(raw_stream):
    if i >= dataset_size:
        break
    data_list.append({
        "question": entry["question"],
        "answer": entry["answer"]["value"],
        "question_id": str(uuid.uuid4())
    })

triviaQA_hf_dataset = Dataset.from_list(data_list)

# 5. Save to Disk
print(f"Saving dataset to: {save_directory}")
triviaQA_hf_dataset.save_to_disk(save_directory)

print("TriviaQA dataset download was completed succesfully")

Streaming TriviaQA from Hugging Face...


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Fetching 1,000 rows...
Saving dataset to: /content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/triviaQA_hf_1k


Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

TriviaQA dataset download was completed succesfully


## Aggregate All Datasets

In [ ]:
from datasets import load_from_disk, concatenate_datasets, Dataset

print("Loading datasets...")
triviaqa = load_from_disk(os.path.join(data_path, "triviaQA_hf_1k"))
ambigqa  = load_from_disk(os.path.join(data_path, "ambigQA_hf_1k"))
abstain  = load_from_disk(os.path.join(data_path, "abstentionBench_hf_1k"))

triviaqa = triviaqa.map(lambda x: {"type": "answer"})
ambigqa  = ambigqa.map(lambda x: {"type": "clarify"})
abstain  = abstain.map(lambda x: {"type": "abstain"})

print("Aggregating datasets...")
combined = concatenate_datasets([triviaqa, ambigqa, abstain])

combined = combined.shuffle(seed=42)

print(f"Total examples: {len(combined)}")
print(f"Breakdown:")
for label in ["answer", "clarify", "abstain"]:
    count = sum(1 for x in combined["type"] if x == label)
    print(f"  {label}: {count}")

output_name = "combined_datasets_hf"
output_dir  = os.path.join(data_path, output_name)
combined.save_to_disk(output_dir)

print(f"Saved aggregated dataset to: {output_dir}")

Loading datasets...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Aggregating datasets...
Total examples: 3000
Breakdown:
  answer: 1000
  clarify: 1000
  abstain: 1000


Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saved aggregated dataset to: /content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/combined_datasets_hf


In [5]:
# LITE verson (max 24 words per question)
import os
import uuid
import numpy as np
import datasets
from datasets import load_dataset, Dataset

TARGET_PER_TYPE = 1_000
MAX_WORDS       = 24
SEED            = 42
TYPE_ORDER      = ["answer", "clarify", "abstain"]

def word_count(text: str) -> int:
    return len(text.strip().split())

def is_short(text: str) -> bool:
    return word_count(text) <= MAX_WORDS

# TriviaQA
print("=" * 60)
print("Streaming TriviaQA (answer)...")

raw_stream  = load_dataset("trivia_qa", "rc.nocontext", split="train", streaming=True)
answer_rows = []
skipped     = 0

for entry in raw_stream:
    if len(answer_rows) >= TARGET_PER_TYPE:
        break
    q = entry["question"]
    if not is_short(q):
        skipped += 1
        continue
    answer_rows.append({
        "question"   : q,
        "answer"     : entry["answer"]["value"],
        "question_id": str(uuid.uuid4()),
        "type"       : "answer",
    })

print(f"  Collected : {len(answer_rows):,}  |  Skipped (> {MAX_WORDS} words): {skipped:,}")

# AmbigQA
print("Streaming AmbigQA (clarify)...")

raw_stream   = load_dataset("ambig_qa", "full", split="train", streaming=True)
clarify_rows = []
skipped_len  = 0
skipped_type = 0

for entry in raw_stream:
    if len(clarify_rows) >= TARGET_PER_TYPE:
        break
    if entry["annotations"]["type"][0] != "multipleQAs":
        skipped_type += 1
        continue
    q = entry["question"]
    if not is_short(q):
        skipped_len += 1
        continue
    clarify_rows.append({
        "question"   : q,
        "answer"     : entry["nq_answer"][0],
        "question_id": str(uuid.uuid4()),
        "type"       : "clarify",
    })

print(f"  Collected : {len(clarify_rows):,}  |  "
      f"Skipped (not multipleQAs): {skipped_type:,}  |  "
      f"Skipped (> {MAX_WORDS} words): {skipped_len:,}")

# AbstentionBench
print("Loading AbstentionBench (abstain)...")

raw_ds       = datasets.load_dataset("facebook/AbstentionBench", trust_remote_code=True)
flat_ds = raw_ds if isinstance(raw_ds, datasets.Dataset) else raw_ds[list(raw_ds.keys())[0]]
flat_ds      = flat_ds.shuffle(seed=SEED)

abstain_rows = []
skipped_flag = 0
skipped_len  = 0

for entry in flat_ds:
    if len(abstain_rows) >= TARGET_PER_TYPE:
        break
    if not entry["should_abstain"]:
        skipped_flag += 1
        continue
    q = entry["question"]
    if not is_short(q):
        skipped_len += 1
        continue
    abstain_rows.append({
        "question"   : q,
        "answer"     : "N/A",
        "question_id": str(uuid.uuid4()),
        "type"       : "abstain",
    })

print(f"  Collected : {len(abstain_rows):,}  |  "
      f"Skipped (should_abstain=False): {skipped_flag:,}  |  "
      f"Skipped (> {MAX_WORDS} words): {skipped_len:,}")

# Validation Check
counts = {"answer": len(answer_rows), "clarify": len(clarify_rows), "abstain": len(abstain_rows)}
for t, n in counts.items():
    if n < TARGET_PER_TYPE:
        raise ValueError(
            f"Not enough '{t}' rows after filtering: got {n}, need {TARGET_PER_TYPE}. "
            "Try a larger source stream or lower MIN_WORDS."
        )

# Interleave combined dataset
print("\nBuilding interleaved combined_datasets_lite_hf ...")

rng = np.random.default_rng(SEED)

def rng_shuffle(lst):
    idx = rng.permutation(len(lst)).tolist()
    return [lst[i] for i in idx]

answer_rows  = rng_shuffle(answer_rows[:TARGET_PER_TYPE])
clarify_rows = rng_shuffle(clarify_rows[:TARGET_PER_TYPE])
abstain_rows = rng_shuffle(abstain_rows[:TARGET_PER_TYPE])

slices = {"answer": answer_rows, "clarify": clarify_rows, "abstain": abstain_rows}

interleaved = []
for i in range(TARGET_PER_TYPE):
    for t in TYPE_ORDER:                  # answer → clarify → abstain
        interleaved.append(slices[t][i])

combined_lite = Dataset.from_list(interleaved)

# Sanity checks
print(f"\nTotal rows     : {len(combined_lite):,}")
print(f"First 6 types  : {combined_lite['type'][:6]}")
print("Breakdown:")
for t in TYPE_ORDER:
    n = sum(1 for x in combined_lite["type"] if x == t)
    print(f"  {t:<10}: {n:,}")

# Verify alternating pattern holds throughout
for i, t in enumerate(combined_lite["type"]):
    expected = TYPE_ORDER[i % 3]
    assert t == expected, f"Pattern broken at index {i}: got '{t}', expected '{expected}'"
print("✓ Alternating pattern verified across all 3,000 rows")

# save
output_dir = os.path.join(data_path, "combined_datasets_lite_hf")
combined_lite.save_to_disk(output_dir)
print(f"\nSaved → {output_dir}")

  Collected : 1,000  |  Skipped (should_abstain=False): 3,466  |  Skipped (> 24 words): 1,936

Building interleaved combined_datasets_lite_hf ...

Total rows     : 3,000
First 6 types  : ['answer', 'clarify', 'abstain', 'answer', 'clarify', 'abstain']
Breakdown:
  answer    : 1,000
  clarify   : 1,000
  abstain   : 1,000
✓ Alternating pattern verified across all 3,000 rows


Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]


Saved → /content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/combined_datasets_lite_hf


## Data Preview

In [7]:
from datasets import load_from_disk

combined = load_from_disk(os.path.join(data_path, "combined_datasets_hf"))

def print_entry(entry, index):
    print(f"  [{index}] Type       : {entry['type']}")
    print(f"       Question   : {entry['question']}")
    print(f"       Answer     : {entry['answer']}")
    print(f"       ID         : {entry['question_id']}")
    print()

print("=" * 70)
print(" PREVIEW: 5 RANDOM SAMPLES")
print("=" * 70)
sample = combined.shuffle(seed=99).select(range(5))
for i, entry in enumerate(sample):
    print_entry(entry, i + 1)

for label in ["answer", "clarify", "abstain"]:
    print("=" * 70)
    print(f" PREVIEW: 5 SAMPLES (type = '{label}')")
    print("=" * 70)
    subset = combined.filter(lambda x: x["type"] == label).shuffle(seed=99).select(range(5))
    for i, entry in enumerate(subset):
        print_entry(entry, i + 1)

 PREVIEW: 5 RANDOM SAMPLES
  [1] Type       : answer
       Question   : Natan Sharansky was released from prison in the USSR to begin a new life where?
       Answer     : Israel
       ID         : dfa327f3-8e7e-4498-a9a9-b158f127d7be

  [2] Type       : answer
       Question   : In which sport did Hollywood star Sonja Henie win Olympic Gold?
       Answer     : Ice Skating
       ID         : a38c0f13-e3ea-40ad-9837-ff5258e6ecc4

  [3] Type       : clarify
       Question   : When did star wars the empire strikes back come out?
       Answer     : 1980
       ID         : a93a3572-b00a-4f20-bbfd-9437778b9077

  [4] Type       : clarify
       Question   : Who played thanos in gardians of the galaxy?
       Answer     : Josh Brolin
       ID         : 59eed05a-7612-4c41-8043-0940ee3f018d

  [5] Type       : abstain
       Question   : Winston has 14 quarters. He then spends half a dollar on candy. How many ?
       Answer     : N/A
       ID         : 4b028b7c-0210-426f-b601-4f8d460

Filter:   0%|          | 0/3000 [00:00<?, ? examples/s]

  [1] Type       : answer
       Question   : What is the native country of Agatha Chrisitie's detective Hercule Poirot?
       Answer     : Belgium
       ID         : ccd5f632-e00e-4567-a948-a0d2f72356cd

  [2] Type       : answer
       Question   : What is the name of the Darth Vader-to-be in the Star Wars Prequel, Episode 1?
       Answer     : Anakin Skywalker
       ID         : b7eb7d2b-7ef6-410e-9299-1704e5bc0d39

  [3] Type       : answer
       Question   : In which year was CNN founded?
       Answer     : 1980
       ID         : f197694a-ceae-4e73-8f0f-0656d4a21f9b

  [4] Type       : answer
       Question   : Which state renewed Mike Tyson's boxing license in 1998?
       Answer     : Nevada
       ID         : 3da88c16-4d43-4321-897e-c29ecf21bb47

  [5] Type       : answer
       Question   : What was the name of Drew Barrymore's character in E.T.?
       Answer     : Gertie
       ID         : 82db6ed1-3ff5-406c-baee-826f349ac905

 PREVIEW: 5 SAMPLES (type = 'clarify'

Filter:   0%|          | 0/3000 [00:00<?, ? examples/s]

  [1] Type       : clarify
       Question   : When did we start to use bc and ad?
       Answer     : in 525
       ID         : fd7917ee-d5e7-44b4-bfe1-e2981dbc9934

  [2] Type       : clarify
       Question   : Who made the first periodic table of elements?
       Answer     : Dmitri Mendeleev
       ID         : b7c4aab2-acf6-476b-86ae-523cc4dea943

  [3] Type       : clarify
       Question   : Location of tigris and euphrates rivers on a map?
       Answer     : Western Asia
       ID         : 8cea375c-7eda-4b4c-8a4f-009bf8bb928a

  [4] Type       : clarify
       Question   : Who played lionel in all in the family?
       Answer     : Michael Jonas Evans
       ID         : e5628c37-437b-404c-8dac-a04c2e561d70

  [5] Type       : clarify
       Question   : What schizophrenic symptoms are decreased by drugs that selectively block the d2 dopamine receptor?
       Answer     : psychotic
       ID         : f3a1f83a-a997-47e1-9dcc-e4cee7360189

 PREVIEW: 5 SAMPLES (type = 'abstai

Filter:   0%|          | 0/3000 [00:00<?, ? examples/s]

  [1] Type       : abstain
       Question   : Why are earthworm bones so soft?
       Answer     : N/A
       ID         : 5eeeb158-da17-4915-ba65-f7c95502fe13

  [2] Type       : abstain
       Question   : Respond to the question using only information given in the context.
Context: Oxygen gas (O
2) can be toxic at elevated partial pressures, leading to convulsions and other health problems.[j] Oxygen toxicity usually begins to occur at partial pressures more than 50 kilopascals (kPa), equal to about 50% oxygen composition at standard pressure or 2.5 times the normal sea-level O
2 partial pressure of about 21 kPa. This is not a problem except for patients on mechanical ventilators, since gas supplied through oxygen masks in medical applications is typically composed of only 30%–50% O
2 by volume (about 30 kPa at standard pressure). (although this figure also is subject to wide variation, depending on type of mask).
Question: What is composed of 21%-50% O2 by volume?
       Answer   

In [8]:
# LITE version
from datasets import load_from_disk

combined = load_from_disk(os.path.join(data_path, "combined_datasets_lite_hf"))

def print_entry(entry, index):
    print(f"  [{index}] Type       : {entry['type']}")
    print(f"       Question   : {entry['question']}")
    print(f"       Answer     : {entry['answer']}")
    print(f"       ID         : {entry['question_id']}")
    print()

print("=" * 70)
print(" PREVIEW: 5 RANDOM SAMPLES")
print("=" * 70)
sample = combined.select(range(5))
for i, entry in enumerate(sample):
    print_entry(entry, i + 1)

for label in ["answer", "clarify", "abstain"]:
    print("=" * 70)
    print(f" PREVIEW: 5 SAMPLES (type = '{label}')")
    print("=" * 70)
    subset = combined.filter(lambda x: x["type"] == label).shuffle(seed=99).select(range(5))
    for i, entry in enumerate(subset):
        print_entry(entry, i + 1)

 PREVIEW: 5 RANDOM SAMPLES
  [1] Type       : answer
       Question   : The actor who played Jack Geller in Friends was married once to which superstar?
       Answer     : Elliott Gould married Barbra Streisand
       ID         : 88e595ba-722f-4666-bfe3-2bf33201c142

  [2] Type       : clarify
       Question   : When will dragon ball super english dub be released?
       Answer     : 2017
       ID         : 6fde1caf-67b9-479b-b017-9c15f6117a0d

  [3] Type       : abstain
       Question   : In interior design what kind of complementary colors are best for
       Answer     : N/A
       ID         : dceab11a-be41-48de-abd5-59266b432b6e

  [4] Type       : answer
       Question   : Golf star Vijay Singh comes form where?
       Answer     : Fiji
       ID         : ac3b0a3b-3194-42c3-9e2b-c3a6a19beee0

  [5] Type       : clarify
       Question   : Where does the grand canal start and finish?
       Answer     : Dublin , in the east
       ID         : f969cef1-e6e7-485f-b700-92559